# AURAVOX — Neural GPU backend (free Colab T4)

This notebook runs the real AI models (MusicGen for music + Bark for vocals) on Colab's **free GPU** and exposes a public URL that the AURAVOX web app can call.

**Steps:**
1. `Runtime → Change runtime type → Hardware accelerator: T4 GPU`, then Save.
2. Run all three cells below (▶ each, or `Runtime → Run all`).
3. Copy the printed **PUBLIC URL** (looks like `https://something.trycloudflare.com`).
4. In the web app, turn on **“Neural (GPU) mode”** and paste that URL, then Generate.

Keep this tab open — closing it (or Colab timing out) stops the server, and you'll get a new URL next time you run it.

In [ ]:
# 1) Install deps (torch/torchaudio are already on Colab; we add the rest)
!pip -q install "transformers>=4.41" scipy fastapi "uvicorn[standard]" nest_asyncio
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — set Runtime to T4 GPU")

In [ ]:
# 2) Download the AURAVOX backend server + the cloudflared tunnel binary
BRANCH = "gh-pages"  # branch that holds ml/colab_server.py
REPO = "k58804494-pixel/ai-song-maker"
!wget -q -O colab_server.py https://raw.githubusercontent.com/{REPO}/{BRANCH}/ml/colab_server.py
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
print("downloaded server + cloudflared")

In [ ]:
# 3) Launch the server + public tunnel, then print the URL to paste into the web app
import subprocess, sys, time, re
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "colab_server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(4)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    if "trycloudflare.com" in line:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
print("\n\n========================================")
print(" PUBLIC URL — paste this into the web app:")
print(" ", url)
print("========================================")
print("First Generate downloads the models (~2-4 GB) and can take a couple of minutes; later ones are fast.")
print("Leave this cell running. Stopping it shuts down the backend.")